# Fitting a neural network by least squares

A port of `NNplotexample.py`.

This is the picture behind the unit's claim that **training an approximation
architecture is a regression problem**. A scalar function is sampled with
noise, a small network is fitted to the samples by minimising

$$\min_{\theta}\ \frac{1}{N}\sum_{i=1}^{N}\big(J_i-\hat J(x_i;\theta)\big)^2,$$

and the fit is compared against the function the samples came from.

Nothing here is reinforcement learning yet. That is the point: the regression
step is a self-contained piece of machinery, and the only thing the rest of
the unit does differently is where the targets $J_i$ come from -- a simulator
rather than a known function.

**Requires `torch`.** Unlike the notebooks of Units 1-11, which need only
`numpy` and `matplotlib`, the Unit 12 notebooks use PyTorch, as the original
scripts do (`pip install torch`).

In [ ]:
import math, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

SEED, HIDDEN, EPOCHS = 42, [64, 64], 400
LR, BATCH, NOISE, SAMPLES = 1e-3, 64, 0.08, 300
DOMAIN = (-1.5, 1.5)

## The target function and the data

The function being regressed is

$$J(x)=\sin(2\pi x)+0.3\,x, \qquad x\in[-1.5,\,1.5],$$

three full periods with a slight tilt, sampled at $300$ uniformly random
points and corrupted with Gaussian noise of standard deviation $0.08$. The
tilt matters: it stops the problem from being symmetric, so a fit that merely
got the period right would still be visibly wrong.

In [ ]:
def target_fn(x):
    return torch.sin(2.0 * math.pi * x) + 0.3 * x

def make_dataset(n_samples=SAMPLES, noise_std=NOISE, domain=DOMAIN):
    xs = torch.from_numpy(np.random.uniform(domain[0], domain[1],
                                            size=(n_samples, 1))).to(torch.float32)
    ys = target_fn(xs).to(torch.float32)
    ys = ys + noise_std * torch.randn_like(ys)
    return TensorDataset(xs, ys)

seed_all(SEED)
ds = make_dataset()
n_val = max(20, int(0.15 * len(ds)))
train_ds, val_ds = torch.utils.data.random_split(ds, [len(ds) - n_val, n_val])
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False)
print(f"{len(train_ds)} training samples, {len(val_ds)} validation samples, "
      f"{len(train_loader)} minibatches per epoch")

## The architecture

Two hidden layers of $64$ units with ReLU between them, then a linear output.
In the notation of the unit this is the structure

$$\hat J(x;\theta)=r^\intercal\sigma\big(Ay(x)+b\big)$$

iterated twice, with $\sigma=\max(\xi,0)$ -- the first of the two activation
functions on the deck's "examples of functions $\sigma$" slide.

Worth keeping in mind while reading the fit below: a ReLU network is
**piecewise linear**. Every curve it draws is a polyline, and the smoothness
you see is a large number of short segments.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim=1, hidden_sizes=HIDDEN, output_dim=1):
        super().__init__()
        layers, last = [], input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(last, h), nn.ReLU()]
            last = h
        layers.append(nn.Linear(last, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = MLP()
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

## Training

Adam on minibatches of $64$, $400$ epochs. Each epoch is four gradient steps
here, so this is the **stochastic gradient** method of the unit: the full
gradient over all $255$ training points is never computed. The next notebook
measures what that is worth.

In [ ]:
def train(model, loader, val_loader=None, epochs=EPOCHS, lr=LR):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    history = []
    for ep in range(1, epochs + 1):
        model.train()
        running = 0.0
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            running += loss.item() * xb.size(0)
        running /= len(loader.dataset)
        val = None
        if val_loader is not None:
            model.eval()
            val = 0.0
            with torch.no_grad():
                for xb, yb in val_loader:
                    val += loss_fn(model(xb), yb).item() * xb.size(0)
            val /= len(val_loader.dataset)
        history.append((ep, running, val))
        if ep % max(1, epochs // 10) == 0 or ep == 1:
            print(f"Epoch {ep:4d}/{epochs}  train_loss={running:.6f}"
                  + (f"  val_loss={val:.6f}" if val is not None else ""))
    return model, history

model, history = train(model, train_loader, val_loader)

## The result

The red curve is the network, the dashed black curve the function the samples
came from, and the blue dots the noisy training data. This is the figure on
the deck's slide 9.

In [ ]:
def plot_results(model, dataset, domain=DOMAIN):
    model.eval()
    xs = np.linspace(domain[0], domain[1], 400).reshape(-1, 1).astype(np.float32)
    with torch.no_grad():
        preds = model(torch.from_numpy(xs)).numpy()
    data_x = torch.stack([t[0] for t in dataset]).numpy()
    data_y = torch.stack([t[1] for t in dataset]).numpy()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(data_x, data_y, s=20, alpha=0.5, label="train (noisy)")
    ax.plot(xs, np.sin(2 * np.pi * xs) + 0.3 * xs, "k--", label="ground truth")
    ax.plot(xs, preds, "r", lw=2, label="model")
    ax.set_xlabel("$x$"); ax.set_ylabel("$J$")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

plot_results(model, ds)

Two things are worth reading off it.

**The fit is good in the middle and poor at the edges.** Near $x=\pm1.5$ the
red curve flattens and departs from the dashed one. There is no mystery: the
domain ends there, so the outermost samples have data on one side only, and a
ReLU network extrapolates by continuing its last linear piece. The deck's
version of this plot shows the same flattening at the right-hand edge.

**The network fits the function, not the noise.** It passes through the middle
of the cloud of blue dots rather than through the dots themselves, which is
what the regression is asking for: the minimiser of the expected squared error
is $\mathbb{E}[J\mid x]$, the conditional mean, and that is the dashed curve.
With $300$ samples and a network this small there is not enough freedom to
chase individual points, which here is a feature.

In [ ]:
final_train = history[-1][1]
final_val   = history[-1][2]
print(f"final train loss {final_train:.6f}")
print(f"final val   loss {final_val:.6f}")
print(f"noise variance   {NOISE**2:.6f}   <- the loss floor: no fit can do better")

That last line is the useful check. The samples carry additive noise of
standard deviation $0.08$, so even a *perfect* fit to $\mathbb{E}[J\mid x]$
would leave a mean squared error of $0.08^2=0.0064$ against the noisy targets.
A training loss near that value means the network has extracted essentially
everything there is; a loss far above it means underfitting, and a loss well
*below* it would mean the network is fitting noise.